# 06 - ReAct Agent

**Learning objective:** See a ReAct agent iterate through Think -> Act -> Observe, calling all five tools, and compare it side-by-side with an ungrounded baseline.

The agent does not answer in a single pass. It plans, then iterates, calling different tools per step, then synthesises the evidence with full source attribution.

In [ ]:
import os
import sys
import subprocess
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
sys.path.insert(0, "..")

# Database connection
import psycopg

pg_dir = "../iac_snippets/postgres"


def get_pg_output(name):
    r = subprocess.run(["tofu", "output", "-raw", name], cwd=pg_dir, capture_output=True, text=True)
    return r.stdout.strip()


conn = psycopg.connect(
    host=get_pg_output("host"),
    port=int(get_pg_output("port")),
    dbname=get_pg_output("database"),
    user=get_pg_output("user"),
    password=get_pg_output("password"),
    sslmode="require",
)

# Embeddings client
mi_dir = "../iac_snippets/managed_inference"
result = subprocess.run(["tofu", "output", "-raw", "endpoint_url"], cwd=mi_dir, capture_output=True, text=True)
endpoint_url = result.stdout.strip()

from src.embeddings import EmbeddingsClient

embedding_client = EmbeddingsClient(
    client=OpenAI(base_url=endpoint_url, api_key=os.environ["SCW_SECRET_KEY"]),
    model="baai/bge-multilingual-gemma2:fp32",
    dimensions=768,
)

# LLM client
llm_client = OpenAI(
    base_url="https://api.scaleway.ai/v1",
    api_key=os.environ["SCW_SECRET_KEY"],
)

# Create toolkit
from src.tools import ToolKit

toolkit = ToolKit(conn=conn, embeddings_client=embedding_client, llm_client=llm_client)

print("All components ready.")

## Baseline: Plain Mistral (no tools)

First, let us see what Mistral says without any tools or grounding.

In [ ]:
CANONICAL_QUERY = (
    "A patient is taking warfarin, aspirin, and ibuprofen. "
    "The patient is 32 weeks pregnant. "
    "What are the drug interactions and population-specific warnings? "
    "List all findings with severity and cite your sources."
)

# Baseline - plain Mistral, no tools
baseline = llm_client.chat.completions.create(
    model="mistral-small-3.2-24b-instruct-2506",
    messages=[{"role": "user", "content": CANONICAL_QUERY}],
    temperature=0.3,
    max_tokens=1000,
)

print("=== BASELINE (plain Mistral, no tools) ===")
print(baseline.choices[0].message.content)

## ReAct Agent: Full Think -> Act -> Observe Trace

Now the same query through the ReAct loop with all five tools.

In [ ]:
from src.react_loop import run_react_loop

result = run_react_loop(
    query=CANONICAL_QUERY,
    toolkit=toolkit,
    llm_client=llm_client,
)

print(f"Agent completed in {len(result['trace'])} iterations.")

In [ ]:
# Render the full trace
print("=== REACT AGENT TRACE ===")
print()
for i, entry in enumerate(result["trace"]):
    print(f"--- Iteration {i + 1} ---")
    print(f"THINK: {entry['think'][:200]}")
    print(f"ACT:   {entry['act'][:150]}")
    observe_preview = entry["observe"][:200]
    print(f"OBSERVE: {observe_preview}...")
    print()

In [ ]:
# Final response
print("=== REACT AGENT FINAL RESPONSE ===")
print(result["final_response"])

## Side-by-Side Comparison

| Aspect | Baseline (plain Mistral) | ReAct Agent |
|--------|-------------------------|-------------|
| Iterations | 1 (single pass) | Multiple Think->Act->Observe |
| Tool calls | 0 | 5+ (lookup_interactions x3, lookup_population_warnings, search_drug_kb, flag_severity, summarize_evidence) |
| Citations | Generic or none | Specific FDA label set_id per claim |
| Severity ordering | Implicit | Explicit CRITICAL -> MAJOR -> MODERATE -> MINOR |
| Verifiable | No | Yes, every claim traces to a specific SPL section |

Grounded, tool-calling agents produce verifiable medical AI output.

## You should now have

- [x] Seen the ReAct agent iterate through multiple tool calls
- [x] All five tools exercised in a single query
- [x] Every finding cites a specific FDA label section by set_id
- [x] Severity-first output (CRITICAL findings first)
- [x] Understood the difference between ungrounded and grounded medical AI

**Next:** `07_deploy_demo_app.ipynb`